## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## Features and split


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
data = spark.table(ns+".day18_shipments")
train = data.filter("shipment_id % 20 < 14")
valid = data.filter("shipment_id % 20 >= 14 AND shipment_id % 20 < 17")
test = data.filter("shipment_id % 20 >= 17")
assert (train.count(),valid.count(),test.count()) == (280,60,60)
for part in [train,valid,test]:
 assert part.select("late").distinct().count() == 2
feature_cols = ["distance_km","items","rain"]
assembler = VectorAssembler(inputCols=feature_cols,outputCol="features")
# Labels, shipment_id and any post-delivery information are excluded.


## Regression


In [ ]:
reg = Pipeline(stages=[assembler,
 LinearRegression(featuresCol="features",labelCol="delivery_hours",
                  regParam=0.1,maxIter=100)])
reg_model = reg.fit(train)
validation_predictions = reg_model.transform(valid)
metric = RegressionEvaluator(labelCol="delivery_hours",predictionCol="prediction")
for name in ["mae","rmse","r2"]:
 print("Regression validation",name,
       metric.setMetricName(name).evaluate(validation_predictions))
mean_hours = train.agg(F.avg("delivery_hours")).first()[0]
baseline = valid.withColumn("prediction",F.lit(mean_hours))
print("Mean baseline validation MAE:",metric.setMetricName("mae").evaluate(baseline))
# Keep these settings fixed before using the final test set.
final_reg_model = reg.fit(train.unionByName(valid))
test_reg = final_reg_model.transform(test)
for name in ["mae","rmse","r2"]:
 print("Regression test",name,metric.setMetricName(name).evaluate(test_reg))
display(test_reg.select("shipment_id","delivery_hours","prediction").orderBy("shipment_id"))
test_reg.select("shipment_id","delivery_hours","prediction").write.format("delta")\
 .mode("overwrite").saveAsTable(ns+".day18_regression_test")


## Classification


In [ ]:
clf = Pipeline(stages=[assembler,
 LogisticRegression(featuresCol="features",labelCol="late",regParam=0.1,maxIter=100)])
clf_model = clf.fit(train)
val_scores = clf_model.transform(valid).withColumn(
 "score",vector_to_array("probability")[1])
def binary_metrics(scored, threshold):
 rows = scored.select("late","score").collect()  # At most 60 tiny lab rows.
 tp=fp=fn=tn=0
 for r in rows:
  actual, predicted = int(r.late), int(r.score >= threshold)
  tp += int(actual==1 and predicted==1)
  fp += int(actual==0 and predicted==1)
  fn += int(actual==1 and predicted==0)
  tn += int(actual==0 and predicted==0)
 precision=tp/(tp+fp) if tp+fp else 0.0
 recall=tp/(tp+fn) if tp+fn else 0.0
 return {"threshold":threshold,"TP":tp,"FP":fp,"FN":fn,"TN":tn,
  "accuracy":(tp+tn)/len(rows),"precision":precision,"recall":recall,
  "F1":2*precision*recall/(precision+recall) if precision+recall else 0.0}
choices=[binary_metrics(val_scores,t) for t in [0.3,0.5,0.7]]
print("Validation thresholds:",choices)
best=max(choices,key=lambda x:(x["F1"],-abs(x["threshold"]-0.5)))
threshold=best["threshold"]
# Keep the trained model and threshold fixed for the final test.
test_scores=clf_model.transform(test).withColumn(
 "score",vector_to_array("probability")[1])
print("Classification final test:",binary_metrics(test_scores,threshold))
evaluator=BinaryClassificationEvaluator(labelCol="late",rawPredictionCol="rawPrediction")
for metric_name in ["areaUnderROC","areaUnderPR"]:
 print(metric_name,evaluator.setMetricName(metric_name).evaluate(test_scores))
output=test_scores.withColumn("predicted_late",(F.col("score")>=threshold).cast("int"))
display(output.select("shipment_id","late","score","predicted_late"))
output.select("shipment_id","late","score","predicted_late").write.format("delta")\
 .mode("overwrite").saveAsTable(ns+".day18_classification_test")
